## Expand Author Profile full_name

Many author profiles have a `full_name` set to a sparse form (e.g. `Louise Jones`, `J Adamski`, `JOHN M. MARTIN`) when a fuller variant — including a middle name or unabbreviated first name — already appears in `raw_author_names` and is consistent with every other variant on the profile.

Because matching uses parsed pieces of `full_name` (`block_key`, `first`, `middle`, `last`) in `authors_for_matching`, a sparse `full_name` discards discriminating signal — making future overmerge across distinct people sharing surname + first-initial more likely.

Examples (current → proposed):
- A5109871191: `Louise Jones` → `Louise Kathleen Jones`
- A5013575867: `J Adamski` → `Jerzy Adamski`
- A5063959638: `JOHN M. MARTIN` → `John Munson Martin`
- A5063002260: `J. H. M. Wöltgens` → `Joseph H. M. Wöltgens`

This is a one-time backfill. Going forward, new authors continue to mint `full_name = display_name` in `MatchAuthors`, and the same logic could be re-run periodically as `raw_author_names` accumulates.

**Detection rules** (per author):
1. Parse `full_name` and every `raw_author_names[]` entry via `openalex.authors.author_names`.
2. Restrict to raw variants whose parsed `last` matches current parsed `last` (after dot-stripping + lowercasing). Drops noise (e.g. a stray `Clay` on a Jones profile).
3. Suffix guardrail: skip authors where any sibling has a non-empty parsed suffix that disagrees with current.
4. String-form filters on the chosen variant: reject variants containing `,` `(` `)` `[` `]` or any digit, and reject all-uppercase variants.
5. Score each remaining variant by `LENGTH(first) + LENGTH(middle)` and take the top-ranked as the proposal.
6. Compatibility check: every other matched-last sibling must agree with the proposal — full-form fields equal, initial fields match the proposal's first letter, and the proposal must have a middle to absorb any sibling's non-empty middle.
7. Emit only when proposal differs from current and is strictly more informative (longer first or longer middle).

**Source of truth**: `openalex.authors.authors.full_name` (the base table). After the MERGE, the next `CreateAuthors` run will rebuild `openalex_authors` and `authors_for_matching` from the new values.

### Cell 1: Build candidates — authors whose full_name can be safely expanded

Sources current `full_name` from `openalex.authors.authors` (truth) and `raw_author_names` from `openalex.authors.openalex_authors` (the aggregated array).

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_fullname_expand_candidates AS
WITH base AS (
  SELECT a.id, a.full_name, oa.raw_author_names
  FROM openalex.authors.authors a
  JOIN openalex.authors.openalex_authors oa ON a.id = oa.id
  WHERE a.full_name IS NOT NULL
    AND oa.raw_author_names IS NOT NULL
    AND SIZE(oa.raw_author_names) > 1
    AND NOT REGEXP_LIKE(a.full_name, '[\\u4e00-\\u9fff\\u3040-\\u30ff\\uac00-\\ud7af\\u0400-\\u04ff]')
),
cur_parsed AS (
  SELECT b.id, b.full_name, b.raw_author_names,
         TRANSLATE(LOWER(pn.parsed_name.first),  '.', '') AS cur_first,
         TRANSLATE(LOWER(pn.parsed_name.middle), '.', '') AS cur_middle,
         TRANSLATE(LOWER(pn.parsed_name.last),   '.', '') AS cur_last,
         TRANSLATE(LOWER(pn.parsed_name.suffix), '.', '') AS cur_suffix
  FROM base b
  LEFT JOIN openalex.authors.author_names pn
    ON TRIM(b.full_name) = pn.raw_author_name
  WHERE pn.parsed_name.last IS NOT NULL AND pn.parsed_name.last <> ''
    AND pn.parsed_name.first IS NOT NULL AND pn.parsed_name.first <> ''
),
exploded AS (
  SELECT c.id, c.full_name, c.cur_first, c.cur_middle, c.cur_last, c.cur_suffix,
         v.variant
  FROM cur_parsed c
  LATERAL VIEW EXPLODE(c.raw_author_names) v AS variant
),
raw_parsed AS (
  SELECT e.id, e.full_name, e.cur_first, e.cur_middle, e.cur_last, e.cur_suffix,
         e.variant,
         TRANSLATE(LOWER(an.parsed_name.first),  '.', '') AS r_first,
         TRANSLATE(LOWER(an.parsed_name.middle), '.', '') AS r_middle,
         TRANSLATE(LOWER(an.parsed_name.last),   '.', '') AS r_last,
         TRANSLATE(LOWER(an.parsed_name.suffix), '.', '') AS r_suffix
  FROM exploded e
  LEFT JOIN openalex.authors.author_names an
    ON TRIM(e.variant) = an.raw_author_name
),
matched_last AS (
  SELECT *
  FROM raw_parsed
  WHERE r_last = cur_last
    AND r_first IS NOT NULL AND r_first <> ''
),
suffix_check AS (
  SELECT id,
         ARRAY_DISTINCT(
           ARRAY_COMPACT(
             ARRAY_UNION(
               ARRAY(NULLIF(ANY_VALUE(cur_suffix), '')),
               COLLECT_SET(NULLIF(r_suffix, ''))
             )
           )
         ) AS distinct_suffixes
  FROM matched_last
  GROUP BY id
),
eligible AS (
  SELECT id FROM suffix_check WHERE SIZE(distinct_suffixes) <= 1
),
candidates AS (
  SELECT ml.*
  FROM matched_last ml
  JOIN eligible e ON ml.id = e.id
  WHERE LENGTH(ml.r_first) >= 2
    AND NOT REGEXP_LIKE(ml.variant, '[,\\(\\)\\[\\]0-9]')
    AND NOT (UPPER(ml.variant) = ml.variant
             AND REGEXP_LIKE(ml.variant, '[A-Za-z]'))
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY id
           ORDER BY LENGTH(COALESCE(r_first, '')) + LENGTH(COALESCE(r_middle, '')) DESC,
                    LENGTH(COALESCE(r_first, '')) DESC,
                    variant ASC
         ) AS rn
  FROM candidates
),
proposal AS (
  SELECT id, full_name, cur_first, cur_middle, cur_last,
         variant   AS proposed_full_name,
         r_first   AS prop_first,
         r_middle  AS prop_middle
  FROM ranked
  WHERE rn = 1
),
compat_check AS (
  SELECT p.id, p.full_name, p.proposed_full_name,
         p.cur_first, p.cur_middle, p.cur_last,
         p.prop_first, p.prop_middle,
         BOOL_AND(
           CASE
             WHEN LENGTH(ml.r_first) >= 2 AND ml.r_first <> p.prop_first THEN false
             WHEN LENGTH(ml.r_first) = 1 AND ml.r_first <> SUBSTRING(p.prop_first, 1, 1) THEN false
             WHEN LENGTH(ml.r_middle) >= 2 AND LENGTH(p.prop_middle) >= 2 AND ml.r_middle <> p.prop_middle THEN false
             WHEN LENGTH(ml.r_middle) = 1 AND LENGTH(p.prop_middle) >= 1 AND ml.r_middle <> SUBSTRING(p.prop_middle, 1, 1) THEN false
             WHEN LENGTH(p.prop_middle) = 0 AND LENGTH(ml.r_middle) >= 1 THEN false
             ELSE true
           END
         ) AS all_compatible,
         COUNT(*) AS sibling_count
  FROM proposal p
  JOIN matched_last ml ON ml.id = p.id
  GROUP BY p.id, p.full_name, p.proposed_full_name,
           p.cur_first, p.cur_middle, p.cur_last,
           p.prop_first, p.prop_middle
)
SELECT
  id            AS author_id,
  full_name     AS current_full_name,
  proposed_full_name AS new_full_name,
  cur_first     AS current_first,
  cur_middle    AS current_middle,
  prop_first    AS new_first,
  prop_middle   AS new_middle,
  sibling_count
FROM compat_check
WHERE all_compatible = true
  AND TRIM(LOWER(proposed_full_name)) <> TRIM(LOWER(full_name))
  AND (LENGTH(prop_first)  > LENGTH(cur_first)
    OR LENGTH(prop_middle) > LENGTH(cur_middle))

### Cell 2: Validation statistics

Expected at time of authoring: ~215K candidates total, dominated by middle-only expansions.

In [ ]:
SELECT
  COUNT(*) AS total_expand_candidates,
  SUM(CASE WHEN LENGTH(new_first) > LENGTH(current_first)
             AND LENGTH(new_middle) > LENGTH(current_middle) THEN 1 ELSE 0 END) AS first_and_middle,
  SUM(CASE WHEN LENGTH(new_first) > LENGTH(current_first)
             AND LENGTH(new_middle) <= LENGTH(current_middle) THEN 1 ELSE 0 END) AS first_only,
  SUM(CASE WHEN LENGTH(new_first) <= LENGTH(current_first)
             AND LENGTH(new_middle) > LENGTH(current_middle) THEN 1 ELSE 0 END) AS middle_only,
  PERCENTILE_APPROX(sibling_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS sibling_count_pctiles
FROM openalex.authors.author_fullname_expand_candidates

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT author_id, current_full_name, new_full_name,
  current_first, new_first, current_middle, new_middle, sibling_count
FROM openalex.authors.author_fullname_expand_candidates
ORDER BY RAND(101)
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_fullname_expand_log AS
SELECT author_id,
  current_full_name AS previous_full_name,
  new_full_name,
  current_first  AS previous_first,
  current_middle AS previous_middle,
  new_first,
  new_middle,
  sibling_count,
  current_timestamp() AS expand_timestamp
FROM openalex.authors.author_fullname_expand_candidates

### Cell 5: Execute the expansion

**WARNING**: This overwrites `full_name` on `openalex.authors.authors` (the source table — downstream `openalex_authors` and `authors_for_matching` rebuild from this). Verify cells 2–3 before running.

After this MERGE, run `CreateAuthors` to refresh `openalex_authors`, `block_key`, and `authors_for_matching`.

In [ ]:
MERGE INTO openalex.authors.authors AS target
USING openalex.authors.author_fullname_expand_candidates AS source
ON target.id = source.author_id
WHEN MATCHED AND target.full_name = source.current_full_name THEN
  UPDATE SET
    target.full_name = source.new_full_name,
    target.updated_date = current_timestamp()

### Cell 6: Post-fix verification

In [ ]:
SELECT
  COUNT(*) AS log_rows,
  SUM(CASE WHEN a.full_name = src.new_full_name THEN 1 ELSE 0 END) AS full_name_matches_new,
  SUM(CASE WHEN a.full_name = src.previous_full_name THEN 1 ELSE 0 END) AS full_name_still_old,
  SUM(CASE WHEN a.full_name <> src.new_full_name
            AND a.full_name <> src.previous_full_name THEN 1 ELSE 0 END) AS full_name_diverged
FROM openalex.authors.authors a
JOIN openalex.authors.author_fullname_expand_log src ON src.author_id = a.id

### Cell 7: Rollback template (use only for systematic FP discovered)

```sql
MERGE INTO openalex.authors.authors AS target
USING openalex.authors.author_fullname_expand_log AS source
  ON target.id = source.author_id
WHEN MATCHED AND target.full_name = source.new_full_name THEN
  UPDATE SET target.full_name = source.previous_full_name
```

Add a `WHERE` clause inside the USING subquery to restrict rollback to a specific class. Don't run as-is unless you intend to reverse all expansions.

### Downstream rebuild

After the MERGE in Cell 5, kick off `notebooks/authors/CreateAuthors.ipynb` so that:
- `openalex.authors.openalex_authors.full_name` and `block_key` reflect the new values
- `openalex.authors.authors_for_matching.first / middle / first_initial` are re-derived
- ES sync picks up the changed `updated_date` on touched authors